In [24]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain.agents import create_agent
from datetime import date

import os

In [34]:
load_dotenv()  # loads .env file into environment variables
api_key = os.getenv("ZEN_API_KEY")

In [35]:
# Initialize the OpenCode Zen model
model = ChatOpenAI(
    model="big-pickle", # Replace with your chosen Zen model id
    #model="nemotron-3-ultra-free",
    openai_api_key=api_key,
    openai_api_base="https://opencode.ai/zen/v1"
)

# Test the connection
# response = model.invoke("Write a quick python function to reverse a string.")
# print(response.content)


## Start by defining the tools that require structured inputs.

In [26]:
@tool
def create_calendar_event(
    title: str,
    start_time: str,       # ISO format: "2024-01-15T14:00:00"
    end_time: str,         # ISO format: "2024-01-15T15:00:00"
    attendees: list[str],  # email addresses
    location: str = ""
) -> str:
    """Create a calendar event. Requires exact ISO datetime format."""
    # Stub: In practice, this would call Google Calendar API, Outlook API, etc.
    return f"Event created: {title} from {start_time} to {end_time} with {len(attendees)} attendees"


@tool
def send_email(
    to: list[str],  # email addresses
    subject: str,
    body: str,
    cc: list[str] = []
) -> str:
    """Send an email via email API. Requires properly formatted addresses."""
    # Stub: In practice, this would call SendGrid, Gmail API, etc.
    return f"Email sent to {', '.join(to)} - Subject: {subject}"


@tool
def get_available_time_slots(
    attendees: list[str],
    date: str,  # ISO format: "2024-01-15"
    duration_minutes: int
) -> list[str]:
    """Check calendar availability for given attendees on a specific date."""
    # Stub: In practice, this would query calendar APIs
    return ["09:00", "14:00", "16:00"]

## we’ll create specialized sub-agents that handle each domain.

In [27]:
CALENDAR_AGENT_PROMPT = (
    f"Today's date is {date.today().isoformat()}."
    "You are a calendar scheduling assistant. "
    "Parse natural language scheduling requests (e.g., 'next Tuesday at 2pm') "
    "into proper ISO datetime formats. "
    "Use get_available_time_slots to check availability when needed. "
    "If there is no suitable time slot, stop and confirm unavailability in your response. "
    "Use create_calendar_event to schedule events. "
    "Always confirm what was scheduled in your final response."
)

calendar_agent = create_agent(
    model,
    tools=[create_calendar_event, get_available_time_slots],
    system_prompt=CALENDAR_AGENT_PROMPT,
)

In [6]:
query = "Schedule a team meeting next Tuesday at 2pm for 1 hour"

stream = calendar_agent.stream_events(
    {"messages": [{"role": "user", "content": query}]},
    version="v3",
)
for kind, item in stream.interleave("messages", "tool_calls"):
    if kind == "messages":
        for token in item.text:
            print(token, end="", flush=True)
    elif kind == "tool_calls":
        print(f"\nTool call: {item.tool_name}({item.input})")
        print(f"Tool result: {item.output}")

/Users/roypsr/projects/gen_ai_mind/.venv/lib/python3.14/site-packages/langgraph/pregel/main.py:3723: LangChainBetaWarning: The v3 streaming protocol on Pregel is experimental.
  return self._pregel_stream_v3(
/Users/roypsr/projects/gen_ai_mind/.venv/lib/python3.14/site-packages/langgraph/pregel/main.py:3573: LangChainBetaWarning: The v3 streaming protocol on Pregel is experimental.
  return GraphRunStream(graph_iter, mux)


I'd be happy to schedule that team meeting for you! First, I need to know who should attend the meeting. Could you please provide the list of attendees (email addresses or names)?

Also, to confirm: "next Tuesday" would be **July 28, 2026** at 2:00 PM for 1 hour (ending at 3:00 PM). Is that correct?

In [7]:
# CALENDAR_AGENT_PROMPT_WITH_TODAY_DATE = (
#     f"Today's date is {date.today().isoformat()}."
#     "You are a calendar scheduling assistant. "
#     "Parse natural language scheduling requests (e.g., 'next Tuesday at 2pm') "
#     "into proper ISO datetime formats. "
#     "Use get_available_time_slots to check availability when needed. "
#     "If there is no suitable time slot, stop and confirm unavailability in your response. "
#     "Use create_calendar_event to schedule events. "
#     "Always confirm what was scheduled in your final response."
# )

# calendar_agent_with_today_date = create_agent(
#     model,
#     tools=[create_calendar_event, get_available_time_slots],
#     system_prompt=CALENDAR_AGENT_PROMPT_WITH_TODAY_DATE,
# )

In [8]:
# query = "Schedule a team meeting next Tuesday at 2pm for 1 hour"

# stream = calendar_agent_with_today_date.stream_events(
#     {"messages": [{"role": "user", "content": query}]},
#     version="v3",
# )
# for kind, item in stream.interleave("messages", "tool_calls"):
#     if kind == "messages":
#         for token in item.text:
#             print(token, end="", flush=True)
#     elif kind == "tool_calls":
#         print(f"\nTool call: {item.tool_name}({item.input})")
#         print(f"Tool result: {item.output}")

In [33]:
#from langchain_core import sys_info
#sys_info.print_sys_info()

In [28]:
EMAIL_AGENT_PROMPT = (
    "You are an email assistant. "
    "Compose professional emails based on natural language requests. "
    "Extract recipient information and craft appropriate subject lines and body text. "
    "If no email addresses are provided, use 'team@company.com' as a placeholder. "
    "Use send_email to send the message. "
    "Always confirm what was sent in your final response."
)

email_agent = create_agent(
    model,
    tools=[send_email],
    system_prompt=EMAIL_AGENT_PROMPT,
)

In [29]:
query = "Send the design team a reminder about reviewing the new mockups"

stream = email_agent.stream_events(
    {"messages": [{"role": "user", "content": query}]},
    version="v3",
)
for kind, item in stream.interleave("messages", "tool_calls"):
    if kind == "messages":
        for token in item.text:
            print(token, end="", flush=True)
    elif kind == "tool_calls":
        print(f"\nTool call: {item.tool_name}({item.input})")
        print(f"Tool result: {item.output}")


Tool call: send_email({'to': ['team@company.com'], 'subject': 'Reminder: Review of New Mockups', 'body': 'Dear Design Team,\n\nThis is a friendly reminder to review the new mockups as soon as possible. Your feedback is crucial to ensure we stay on schedule and maintain the quality of our design work.\n\nPlease take some time to examine the mockups and share your comments or suggestions by the end of this week.\n\nThank you for your attention to this matter.\n\nBest regards,\n[Your Name]', 'cc': []})
Tool result: None
Email sent to the design team (team@company.com) with subject "Reminder: Review of New Mockups." The message reminded the team to review the new mockups and provide feedback by the end of the week.

Wrap the sub-agents with tools

In [30]:
@tool
def schedule_event(request: str) -> str:
    """Schedule calendar events using natural language.

    Use this when the user wants to create, modify, or check calendar appointments.
    Handles date/time parsing, availability checking, and event creation.

    Input: Natural language scheduling request (e.g., 'meeting with design team
    next Tuesday at 2pm')
    """
    result = calendar_agent.invoke({
        "messages": [{"role": "user", "content": request}]
    })
    return result["messages"][-1].text


@tool
def manage_email(request: str) -> str:
    """Send emails using natural language.

    Use this when the user wants to send notifications, reminders, or any email
    communication. Handles recipient extraction, subject generation, and email
    composition.

    Input: Natural language email request (e.g., 'send them a reminder about
    the meeting')
    """
    result = email_agent.invoke({
        "messages": [{"role": "user", "content": request}]
    })
    return result["messages"][-1].text

In [31]:
SUPERVISOR_PROMPT = (
    "You are a helpful personal assistant. "
    "You can schedule calendar events and send emails. "
    "Break down user requests into appropriate tool calls and coordinate the results. "
    "When a request involves multiple actions, use multiple tools in sequence or in parallel as appropriate."
)

supervisor_agent = create_agent(
    model,
    tools=[schedule_event, manage_email],
    system_prompt=SUPERVISOR_PROMPT,
)

## single multi-domain request

In [52]:
# query = (
#     "Schedule a meeting with the design team next Tuesday at 2pm for 1 hour, "
#     "and send them an email reminder about reviewing the new mockups."
# )
query = "Schedule a team standup for tomorrow at 9am"


stream = supervisor_agent.stream_events(
    {"messages": [{"role": "user", "content": query}]},
    version="v3",
)
for kind, item in stream.interleave("messages", "tool_calls"):
    if kind == "messages":
        for token in item.text:
            print(token, end="", flush=True)
    elif kind == "tool_calls":
        print(f"\nTool call: {item.tool_name}({item.input})")
        print(f"Tool result: {item.output}")


Tool call: schedule_event({'request': 'team standup tomorrow at 9am'})
Tool result: None
I'd be happy to schedule the team standup for tomorrow at 9am! To complete the scheduling, I need a couple of details:

1. **Attendees**: Who should be invited? Please provide email addresses or names.
2. **Duration**: How long should the standup be? (Typically 15 minutes for a standup)

Once you provide the attendee list, I'll check availability for tomorrow (2026-07-27) at 9:00 AM and schedule the event if there are no conflicts.

## Add human in loop

In [32]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware 
from langgraph.checkpoint.memory import InMemorySaver 


calendar_agent = create_agent(
    model,
    tools=[create_calendar_event, get_available_time_slots],
    system_prompt=CALENDAR_AGENT_PROMPT,
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={"create_calendar_event": True},
            description_prefix="Calendar event pending approval",
        ),
    ],
)

email_agent = create_agent(
    model,
    tools=[send_email],
    system_prompt=EMAIL_AGENT_PROMPT,
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={"send_email": True},
            description_prefix="Outbound email pending approval",
        ),
    ],
)

supervisor_agent = create_agent(
    model,
    tools=[schedule_event, manage_email],
    system_prompt=SUPERVISOR_PROMPT,
    checkpointer=InMemorySaver(),
)

In [33]:
query = (
    "Schedule a meeting with the design team next Tuesday at 2pm for 1 hour, "
    "and send them an email reminder about reviewing the new mockups."
)

config = {"configurable": {"thread_id": "6"}}

interrupts = []
stream = supervisor_agent.stream_events(
    {"messages": [{"role": "user", "content": query}]},
    config,
    version="v3",
)
for kind, item in stream.interleave("messages", "tool_calls"):
    if kind == "messages":
        for token in item.text:
            print(token, end="", flush=True)
    elif kind == "tool_calls":
        print(f"\nTool call: {item.tool_name}({item.input})")
if stream.interrupted:
    for interrupt_ in stream.interrupts:
        interrupts.append(interrupt_)
        print(f"\nINTERRUPTED: {interrupt_.id}")

I'll schedule the meeting and send the email reminder at the same time since these are independent actions.
Tool call: schedule_event({'request': 'Meeting with the design team next Tuesday at 2pm for 1 hour'})

Tool call: manage_email({'request': 'Send an email reminder to the design team about reviewing the new mockups for our upcoming meeting'})

INTERRUPTED: 1c324b3e133501bf801e1a67b88df56c

INTERRUPTED: 888393c1f4bacbfbb36019f6668d81b1


In [17]:
stream.interrupts

[Interrupt(value={'action_requests': [{'name': 'create_calendar_event', 'args': {'title': 'Meeting with the design team', 'start_time': '2026-07-28T14:00:00', 'end_time': '2026-07-28T15:00:00', 'attendees': ['design team']}, 'description': "Calendar event pending approval\n\nTool: create_calendar_event\nArgs: {'title': 'Meeting with the design team', 'start_time': '2026-07-28T14:00:00', 'end_time': '2026-07-28T15:00:00', 'attendees': ['design team']}"}], 'review_configs': [{'action_name': 'create_calendar_event', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']}]}, id='2c26a3240ace989140ffd17aa4f019d9')]

In [12]:
interrupts

[Interrupt(value={'action_requests': [{'name': 'create_calendar_event', 'args': {'title': 'Meeting with the design team', 'start_time': '2026-07-28T14:00:00', 'end_time': '2026-07-28T15:00:00', 'attendees': ['design team']}, 'description': "Calendar event pending approval\n\nTool: create_calendar_event\nArgs: {'title': 'Meeting with the design team', 'start_time': '2026-07-28T14:00:00', 'end_time': '2026-07-28T15:00:00', 'attendees': ['design team']}"}], 'review_configs': [{'action_name': 'create_calendar_event', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']}]}, id='2c26a3240ace989140ffd17aa4f019d9')]

In [20]:
resume = {}
resume["2c26a3240ace989140ffd17aa4f019d9"] = {"decisions": [{"type": "approve"}]}

In [22]:
from langgraph.types import Command 

stream = supervisor_agent.stream_events(
    Command(resume=resume),
    config,
    version="v3",
)
for kind, item in stream.interleave("messages", "tool_calls"):
    if kind == "messages":
        for token in item.text:
            print(token, end="", flush=True)
    elif kind == "tool_calls":
        print(f"\nTool call: {item.tool_name}({item.input})")
if stream.interrupted:
    for interrupt_ in stream.interrupts:
        interrupts.append(interrupt_)
        print(f"\nINTERRUPTED: {interrupt_.id}")


Tool call: schedule_event({'request': 'Meeting with the design team next Tuesday at 2pm for 1 hour'})
Great news! Here's the status:

✅ **Meeting Scheduled:**
- **Title:** Meeting with the design team
- **Date:** Tuesday, July 28, 2026
- **Time:** 2:00 PM - 3:00 PM (1 hour)
- **Attendees:** Design team

⏳ **Email Reminder:** I need a bit more information to send the email. Could you please provide:
- The email address(es) for the design team (e.g., `design-team@company.com` or individual addresses)

Once you share that, I'll send them a reminder about reviewing the new mockups!

In [56]:
query = (
    "Schedule a meeting with the design team next Tuesday at 2pm for 1 hour, "
    "and send them an email reminder about reviewing the new mockups."
)

config = {"configurable": {"thread_id": "6"}}

interrupts = []
stream = supervisor_agent.stream_events(
    {"messages": [{"role": "user", "content": query}]},
    config,
    version="v3",
)
for kind, item in stream.interleave("messages", "tool_calls"):
    if kind == "messages":
        for token in item.text:
            print(token, end="", flush=True)
    elif kind == "tool_calls":
        print(f"\nTool call: {item.tool_name}({item.input})")
if stream.interrupted:
    for interrupt_ in stream.interrupts:
        interrupts.append(interrupt_)
        print(f"\nINTERRUPTED: {interrupt_.id}")
        for request in interrupt_.value["action_requests"]:
            print(f"{request['description']}\n")
            print(f"Tool: {request['tool']}")
            print(f"Args: {request['args']}\n")

# Resume after approval to continue to the next tool call (manage_email)
if interrupts:
    for interrupt_ in interrupts:
        for request in interrupt_.value["action_requests"]:
            # In real usage, you'd get human approval here
            approved = True  # auto-approve for demo
            if approved:
                supervisor_agent.resume(
                    config,
                    interrupt_.id,
                    {"action": "approve", "args": request["args"]}
                )
    
    # Resume the stream to continue to manage_email
    resume_stream = supervisor_agent.stream_events(
        None,  # None resumes from interrupt
        config,
        version="v3",
    )
    for kind, item in resume_stream.interleave("messages", "tool_calls"):
        if kind == "messages":
            for token in item.text:
                print(token, end="", flush=True)
        elif kind == "tool_calls":
            print(f"\nTool call: {item.tool_name}({item.input})")
    if resume_stream.interrupted:
        for interrupt_ in resume_stream.interrupts:
            interrupts.append(interrupt_)
            print(f"\nINTERRUPTED: {interrupt_.id}")
            for request in interrupt_.value["action_requests"]:
                print(f"{request['description']}\n")
                print(f"Tool: {request['tool']}")
                print(f"Args: {request['args']}\n")


Tool call: manage_email({'request': 'send an email reminder to the design team about reviewing the new mockups'})
I've scheduled the meeting with the design team for next Tuesday at 2pm for 1 hour. 

For the email reminder about reviewing the new mockups, I need the design team's email addresses to send it. Could you please provide:

1. The email addresses for the design team members (or a group email address if you have one)
2. Any specific details you'd like included in the reminder (such as a deadline, which specific mockups to review, links to the files, etc.)

Once I have this information, I'll send the reminder email right away.

In [55]:
# Interrupt handling and resume now done in cell 54 above

INTERRUPTED: 1fc258aca148bfa791492d7a6f1425d2
Calendar event pending approval

Tool: create_calendar_event
Args: {'title': 'Meeting with design team', 'attendees': ['design-team@company.com'], 'end_time': '2026-07-28T15:00:00', 'start_time': '2026-07-28T14:00:00'}

